# Golden Set Creation - 5 Row Sample

This notebook is a hard-coded runtime sample from `master_results_with_submit_edited_0503.xlsx`. It writes fresh golden reference JSON files for the sampled request IDs into `golden_sample_0503/`.

In [ ]:
from pathlib import Path
from queue import Queue

from eval_runner import DEFAULT_CONNECTION_STRINGS
from golden_set_writer import GoldenSetWriter
from metric_manager import MetricManager
from query_runner import ParallelQueryRunner


In [ ]:
SAMPLE_RECORDS = [
    {
        "request_id": 25,
        "db_id": "Insurance",
        "question": "How many beneficiaries appear in inpatient claims but have no outpatient claims?",
        "golden_sql": "select count(distinct [DESYNPUF_ID]) as beneficiary from ( select [DESYNPUF_ID] from [dbo].[Inpatient_Claim] except select [DESYNPUF_ID] from [dbo].[Outpatient_Claim] ) as inpatient_only;",
        "models": {
            "gpt4o": "SELECT COUNT(DISTINCT b.DESYNPUF_ID) AS BeneficiaryCount\nFROM beneficiary b\nJOIN inpatient_claim ic ON b.DESYNPUF_ID = ic.DESYNPUF_ID\nLEFT JOIN outpatient_claim oc ON b.DESYNPUF_ID = oc.DESYNPUF_ID\nWHERE oc.DESYNPUF_ID IS NULL;",
            "claude": "SELECT COUNT(DISTINCT ic.DESYNPUF_ID) AS beneficiary_count\nFROM inpatient_claim ic\nWHERE ic.DESYNPUF_ID NOT IN (\n    SELECT DISTINCT DESYNPUF_ID\n    FROM outpatient_claim\n)",
            "gemini": "SELECT COUNT(DISTINCT T1.DESYNPUF_ID) FROM inpatient_claim AS T1 LEFT JOIN outpatient_claim AS T2 ON T1.DESYNPUF_ID = T2.DESYNPUF_ID WHERE T2.DESYNPUF_ID IS NULL"
        }
    },
    {
        "request_id": 29,
        "db_id": "Insurance",
        "question": "In 2010, how many beneficiaries meet at least one of these conditions:(1) Has diabetes and COPD, or (2) Has depression but no heart failure?",
        "golden_sql": "select count(distinct [DESYNPUF_ID]) as beneficiary from [dbo].[Beneficiary] where [YEAR]=2010 and ( ([SP_DIABETES]=1 and [SP_COPD]=1) or ([SP_DEPRESSN]=1 and [SP_CHF]=0) );",
        "models": {
            "gpt4o": "SELECT COUNT(DISTINCT b.DESYNPUF_ID) \nFROM beneficiary b \nWHERE b.YEAR = 2010 \nAND ((b.SP_DIABETES = 1 AND b.SP_COPD = 1) \nOR (b.SP_DEPRESSN = 1 AND b.SP_CHF = 2))",
            "claude": "\nSELECT COUNT(DISTINCT DESYNPUF_ID) AS beneficiary_count\nFROM beneficiary\nWHERE YEAR = 2010\n  AND (\n    (SP_DIABETES = 1 AND SP_COPD = 1)\n    OR (SP_DEPRESSN = 1 AND SP_CHF = 2)\n  )\n",
            "gemini": "SELECT COUNT(DISTINCT DESYNPUF_ID) FROM beneficiary WHERE YEAR = 2010 AND ((SP_DIABETES = 1 AND SP_COPD = 1) OR (SP_DEPRESSN = 1 AND SP_CHF = 2))"
        }
    },
    {
        "request_id": 93,
        "db_id": "GreaterManchesterCrime",
        "question": "What time do most of the crimes happen?",
        "golden_sql": "SELECT TOP (1) CrimeTS FROM GreaterManchesterCrime GROUP BY CrimeTS ORDER BY count(*) DESC;",
        "models": {
            "gpt4o": "SELECT TOP 1 CrimeTS, COUNT(*) AS CrimeCount\nFROM GreaterManchesterCrime\nGROUP BY CrimeTS\nORDER BY CrimeCount DESC;",
            "claude": "SELECT TOP 1 CrimeTS, COUNT(*) AS CrimeCount\nFROM GreaterManchesterCrime\nWHERE CrimeTS IS NOT NULL\nGROUP BY CrimeTS\nORDER BY CrimeCount DESC",
            "gemini": "SELECT TOP 1 Type, COUNT(*) FROM GreaterManchesterCrime GROUP BY Type ORDER BY COUNT(*) DESC"
        }
    },
    {
        "request_id": 166,
        "db_id": "TheHistoryofBaseball",
        "question": "Top 10 teams with the most hall of fame players",
        "golden_sql": "SELECT TOP (10) T2.team_id FROM hall_of_fame as T1 JOIN salary as T2 ON T1.player_id = T2.player_id AND T1.yearid = T2.year WHERE T1.inducted = 'Y' GROUP BY T2.team_id ORDER BY count(*) DESC;",
        "models": {
            "gpt4o": "SELECT TOP 10 s.team_id, COUNT(h.player_id) AS hall_of_fame_count\nFROM hall_of_fame h\nJOIN salary s ON h.player_id = s.player_id\nGROUP BY s.team_id\nORDER BY hall_of_fame_count DESC;",
            "claude": "SELECT TOP 10\n    s.team_id,\n    COUNT(DISTINCT h.player_id) AS hof_player_count\nFROM hall_of_fame h\nINNER JOIN salary s ON h.player_id = s.player_id\nWHERE h.inducted = 'Y'\nGROUP BY s.team_id\nORDER BY hof_player_count DESC",
            "gemini": "SELECT TOP 10 T2.team_id ,  count(*) FROM hall_of_fame AS T1 JOIN salaries AS T2 ON T1.player_id = T2.player_id GROUP BY T2.team_id ORDER BY count(*) DESC"
        }
    },
    {
        "request_id": 237,
        "db_id": "WhatCDHipHop",
        "question": "Find me the most popular topics since 2010.",
        "golden_sql": "SELECT TOP (10) T1.tag FROM torrents as T2 JOIN tags as T1 ON T1.id = T2.id WHERE T2.groupYear >= 2010 GROUP BY T1.tag ORDER BY T2.totalSnatched DESC;",
        "models": {
            "gpt4o": "SELECT tag, COUNT(*) AS popularity\nFROM tags\nJOIN torrents ON tags.id = torrents.groupId\nWHERE torrents.groupYear >= 2010\nGROUP BY tag\nORDER BY popularity DESC;",
            "claude": "SELECT TOP 10 t.tag, COUNT(*) as frequency\nFROM tags t\nINNER JOIN torrents tr ON t.index = tr.id\nWHERE tr.groupYear >= 2010\nGROUP BY t.tag\nORDER BY frequency DESC",
            "gemini": "SELECT TOP 10 groupName, totalSnatched FROM torrents ORDER BY totalSnatched DESC"
        }
    }
]

[(r["request_id"], r["db_id"], r["question"]) for r in SAMPLE_RECORDS]


In [ ]:
GOLDEN_OUT_DIR = "golden_sample_0503"
GOLDEN_INDEX_CSV = "golden_sample_index_0503.csv"

Path(GOLDEN_INDEX_CSV).write_text("", encoding="utf-8")


def run_golden_sample(records):
    events_q = Queue()
    mm = MetricManager(
        writer_queue=events_q,
        enable_fact_writes=True,
        enable_metric_writes=False,
    )
    golden_writer = GoldenSetWriter(
        event_queue=events_q,
        out_dir=GOLDEN_OUT_DIR,
        index_csv_path=GOLDEN_INDEX_CSV,
        request_id_key="request_id",
        overwrite_json=True,
    )
    golden_writer.start()

    for db_id in sorted({r["db_id"] for r in records}):
        db_records = [r for r in records if r["db_id"] == db_id]
        runner = ParallelQueryRunner(
            connection_string=DEFAULT_CONNECTION_STRINGS[db_id],
            num_workers=2,
            result_callback=mm.add_result,
        )
        runner.start()
        for r in db_records:
            runner.submit(
                r["golden_sql"],
                metadata={
                    "run": "sample_0503",
                    "model": "golden",
                    "request_id": r["request_id"],
                    "db_id": r["db_id"],
                    "question": r["question"],
                },
            )
        runner.stop(drain=True)

    mm.flush_writes()
    golden_writer.stop(drain=True)
    return golden_writer.get_stats()


golden_stats = run_golden_sample(SAMPLE_RECORDS)
golden_stats
